<a href="https://colab.research.google.com/github/elitanima/Custom_AI_Model/blob/main/AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

In [ ]:
# Pip install method (recommended)

!pip install ultralytics

from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="Ключ")
project = rf.workspace("tets1-k0emt").project("project_test_1-qlnqb")
version = project.version(3)
dataset = version.download("yolov8")

In [ ]:
import os
import shutil
import yaml
from pathlib import Path
from sklearn.model_selection import train_test_split

print("=" * 70)
print("📊 ОПТИМАЛЬНОЕ РАЗДЕЛЕНИЕ: 80% TRAIN / 20% VALID")
print("=" * 70)

# ====== КОНФИГУРАЦИЯ ======
dataset_path = '/content/Project_Test_1-3'
data_yaml = os.path.join(dataset_path, 'data.yaml')

# ====== ПОИСК ИЗОБРАЖЕНИЙ ======
print("\n🔍 Поиск изображений...")
train_img_dir = os.path.join(dataset_path, 'train', 'images')
train_lbl_dir = os.path.join(dataset_path, 'train', 'labels')

if not os.path.exists(train_img_dir):
    print("❌ Папка train/images не существует!")
    print("\n🔎 Поиск изображений...")
    for root, dirs, files in os.walk(dataset_path):
        images = [f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if images:
            print(f"   Найдено {len(images)} изображений в: {root}")
            train_img_dir = root
            train_lbl_dir = root.replace('images', 'labels')
            if not os.path.exists(train_lbl_dir):
                train_lbl_dir = os.path.join(os.path.dirname(root), 'labels')
            break

# Получаем список всех изображений
all_images = [f for f in os.listdir(train_img_dir)
              if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

print(f"\n📸 Найдено изображений: {len(all_images)}")

if len(all_images) == 0:
    print("❌ ОШИБКА: Нет изображений для разделения!")
    exit()

# ====== РАЗДЕЛЕНИЕ: 80/20 ======
print("\n🔀 Разделение датасета...")
print("   📊 Train: 80%")
print("   📊 Valid: 20%")
print("   📊 Test:  20% (используем valid как test)")

# Простое разделение 80/20
train_imgs, valid_imgs = train_test_split(
    all_images,
    test_size=0.20,  # 20% в valid
    random_state=42,
    shuffle=True
)

# Test = Valid (используем те же данные)
test_imgs = valid_imgs.copy()

print(f"\n✅ Разделение выполнено:")
print(f"   Train: {len(train_imgs)} изображений ({len(train_imgs)/len(all_images)*100:.1f}%)")
print(f"   Valid: {len(valid_imgs)} изображений ({len(valid_imgs)/len(all_images)*100:.1f}%)")
print(f"   Test:  {len(test_imgs)} изображений ({len(test_imgs)/len(all_images)*100:.1f}%) [= Valid]")

# ====== СОЗДАНИЕ ПАПОК ======
print("\n📁 Создание структуры папок...")

splits = {
    'train': train_imgs,
    'valid': valid_imgs,
    'test': test_imgs
}

for split_name, img_list in splits.items():
    split_img_dir = os.path.join(dataset_path, split_name, 'images')
    split_lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    os.makedirs(split_img_dir, exist_ok=True)
    os.makedirs(split_lbl_dir, exist_ok=True)

    print(f"   ✅ {split_name}/images")
    print(f"   ✅ {split_name}/labels")

# ====== ПЕРЕМЕЩЕНИЕ ФАЙЛОВ ======
print("\n🚚 Перемещение файлов...")

# Создаем временные папки
temp_img_dir = os.path.join(dataset_path, '_temp_images')
temp_lbl_dir = os.path.join(dataset_path, '_temp_labels')

# Копируем все в temp
if os.path.exists(train_img_dir):
    shutil.copytree(train_img_dir, temp_img_dir, dirs_exist_ok=True)
if os.path.exists(train_lbl_dir):
    shutil.copytree(train_lbl_dir, temp_lbl_dir, dirs_exist_ok=True)

# Очищаем старые папки train
for folder in [train_img_dir, train_lbl_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

# Распределяем файлы
for split_name, img_list in splits.items():
    split_img_dir = os.path.join(dataset_path, split_name, 'images')
    split_lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    for img_name in img_list:
        base_name = os.path.splitext(img_name)[0]
        lbl_name = base_name + '.txt'

        # Источники (из temp)
        src_img = os.path.join(temp_img_dir, img_name)
        src_lbl = os.path.join(temp_lbl_dir, lbl_name)

        # Назначения
        dst_img = os.path.join(split_img_dir, img_name)
        dst_lbl = os.path.join(split_lbl_dir, lbl_name)

        # Копируем
        if os.path.exists(src_img):
            shutil.copy2(src_img, dst_img)

        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)

    print(f"   ✅ {split_name}: {len(img_list)} файлов перемещено")

# Удаляем temp папки
shutil.rmtree(temp_img_dir, ignore_errors=True)
shutil.rmtree(temp_lbl_dir, ignore_errors=True)

# ====== ОБНОВЛЕНИЕ data.yaml ======
print("\n📝 Обновление data.yaml...")

with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

# Обновляем пути (абсолютные)
config['path'] = dataset_path
config['train'] = f'{dataset_path}/train/images'
config['val'] = f'{dataset_path}/valid/images'
config['test'] = f'{dataset_path}/test/images'

# Сохраняем
with open(data_yaml, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("   ✅ data.yaml обновлен")

# ====== ФИНАЛЬНАЯ ПРОВЕРКА ======
print("\n" + "=" * 70)
print("✅ ИТОГОВАЯ СТРУКТУРА")
print("=" * 70)

print(f"\n📄 Конфигурация data.yaml:")
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

print("\n📊 Статистика по splits:")
for split_name in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, split_name, 'images')
    lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    img_count = len([f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(img_dir) else 0
    lbl_count = len([f for f in os.listdir(lbl_dir)
                    if f.lower().endswith('.txt')]) if os.path.exists(lbl_dir) else 0

    status = "✅" if img_count > 0 else "❌"
    print(f"{status} {split_name.upper():6} → Images: {img_count:4} | Labels: {lbl_count:4}")

print("\n" + "=" * 70)
print("🚀 ГОТОВО! ЗАПУСКАЙ ОБУЧЕНИЕ")
print("=" * 70)

In [ ]:
import yaml
import os

# ====== ПРОВЕРКА data.yaml ======
dataset_path = '/content/Project_Test_1-3/data.yaml'
base_path = '/content/Project_Test_1-3'

# Читаем конфиг
with open(dataset_path, 'r') as f:
    config = yaml.safe_load(f)

print("=" * 70)
print("📊 ПРОВЕРКА ДАТАСЕТА")
print("=" * 70)

print(f"\n📄 Содержимое data.yaml:")
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

print("\n📊 Проверка папок:")
for split in ['train', 'valid', 'test']:
    img_path = f'{base_path}/{split}/images'
    lbl_path = f'{base_path}/{split}/labels'

    if os.path.exists(img_path):
        img_count = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))])
        lbl_count = len([f for f in os.listdir(lbl_path) if f.endswith('.txt')])
        print(f"   ✅ {split.upper()}: {img_count} images, {lbl_count} labels")
    else:
        print(f"   ❌ {split.upper()}: папка не найдена!")

print("\n" + "=" * 70)
print("🚀 ВСЁ ГОТОВО К ОБУЧЕНИЮ!")
print("=" * 70)

In [ ]:
from ultralytics import YOLO

print("=" * 70)
print("🚀 ОБУЧЕНИЕ YOLOV8L ДЛЯ АВТОКАД ЧЕРТЕЖЕЙ")
print("=" * 70)

# ====== ЗАГРУЗКА МОДЕЛИ ======
model = YOLO('yolov8l.pt')  # ← YOLOv8l - оптимальный баланс
print("✅ Загружена YOLOv8l (оптимальная модель)")

# ====== ОБУЧЕНИЕ ДЛЯ АВТОКАД ЧЕРТЕЖЕЙ ======
print("\n📊 Параметры обучения:")
print("   Epochs: 150")
print("   Image size: 1280")
print("   Batch: 8")
print("   Optimizer: AdamW")
print("   Augmentations: Оптимизированы для AutoCAD")

print("\n" + "=" * 70)
print("⏳ НАЧИНАЕМ ОБУЧЕНИЕ...")
print("=" * 70 + "\n")

results = model.train(
    data='/content/Project_Test_1-3/data.yaml',

    # ========== ОСНОВНЫЕ ПАРАМЕТРЫ ==========
    epochs=150,              # ✅ Увеличено для лучшей сходимости
    imgsz=1280,              # ✅ Оптимально для Tesla T4
    batch=8,                 # ✅ Максимум для вашей GPU
    device=0,

    # ========== ВАЛИДАЦИЯ И СОХРАНЕНИЕ ==========
    val=True,
    patience=30,             # ✅ Увеличено (было 20)
    save=True,
    save_period=10,

    # ========== ОПТИМИЗАТОР ==========
    optimizer='AdamW',
    lr0=0.0005,              # ✅ Стандартный LR
    lrf=0.0001,              # ✅ Низкий финальный LR
    momentum=0.937,
    weight_decay=0.001,      # ✅ Увеличена регуляризация
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # ========== LOSS WEIGHTS ==========
    box=7.5,                 # ✅ Увеличен вес box loss
    cls=0.5,
    dfl=1.5,

    # ========== АУГМЕНТАЦИИ ДЛЯ AUTOCAD ==========
    # Цветовые вариации (разные экспорты PDF/PNG)
    hsv_h=0.02,              # ✅ Вариации оттенка
    hsv_s=0.6,               # ✅ Разная насыщенность слоев
    hsv_v=0.6,               # ✅ Разная яркость экспорта

    # Геометрические трансформации
    degrees=0,               # ❌ Чертежи всегда прямо
    translate=0.3,           # ✅✅✅ КРИТИЧНО! Чертежи в любом месте
    scale=0.6,               # ✅✅✅ КРИТИЧНО! Разные масштабы
    shear=0,                 # ❌ Не искажаем
    perspective=0.0,         # ❌ Не нужна перспектива

    # Отражения
    flipud=0.0,              # ❌ Чертежи не переворачиваются
    fliplr=0.0,              # ❌ Зеркало нарушает читаемость

    # Композиция (имитация плотной компоновки)
    mosaic=1.0,              # ✅✅✅ КРИТИЧНО! Имитирует несколько чертежей
    mixup=0.0,               # ❌ Не подходит для чертежей
    copy_paste=0.3,          # ✅ Имитирует разные расположения

    # ========== ПРОДВИНУТЫЕ НАСТРОЙКИ ==========
    close_mosaic=30,         # ✅ Отключаем mosaic за 30 эпох до конца
    amp=True,                # ✅ Mixed Precision
    fraction=1.0,            # Использовать 100% данных

    # Многомасштабное обучение
    multi_scale=True,        # ✅ Тренировка на разных размерах

    # Для перекрывающихся объектов
    overlap_mask=True,       # ✅ Лучше для плотных компоновок
    mask_ratio=4,

    # ========== ВИЗУАЛИЗАЦИЯ ==========
    plots=True,
    verbose=True,

    # ========== НАЗВАНИЕ ПРОЕКТА ==========
    project='runs/detect',
    name='yolov8l_autocad_80_20',
    exist_ok=False,
)

print("\n" + "=" * 70)
print("🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("=" * 70)
print(f"📁 Результаты: {results.save_dir}")
print(f"🏆 Лучшая модель: {results.save_dir}/weights/best.pt")
print(f"📊 Последняя модель: {results.save_dir}/weights/last.pt")

# ====== ВАЛИДАЦИЯ ЛУЧШЕЙ МОДЕЛИ ======
print("\n" + "=" * 70)
print("🔍 ВАЛИДАЦИЯ ЛУЧШЕЙ МОДЕЛИ")
print("=" * 70 + "\n")

best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val()

print(f"\n📊 Финальные метрики:")
print(f"   mAP50:     {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)")
print(f"   mAP50-95:  {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)")
print(f"   Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)")
print(f"   Recall:    {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)")

print("\n✅ ГОТОВО! Используйте best.pt для детекции чертежей.")

In [ ]:
from ultralytics import YOLO
from google.colab import files

# Экспорт в ONNX
model = YOLO('/content/runs/detect/runs/detect/blueprint_yolov8x/weights/best.pt')
onnx_path = model.export(format='onnx', imgsz=1280, simplify=True)

# Скачивание
files.download('/content/runs/detect/runs/detect/blueprint_yolov8x/weights/best.pt')
files.download('/content/runs/detect/runs/detect/blueprint_yolov8x/weights/best.onnx')

print("✅ Модели скачаны!")

In [ ]:
from IPython.display import Image, display
import pandas as pd
import os

print("="*70)
print("📊 ПОЛНАЯ ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ОБУЧЕНИЯ")
print("="*70)

# ПРАВИЛЬНЫЙ путь (двойной, как на скриншоте)
results_path = '/content/runs/detect/runs/detect/blueprint_yolov8x'

# 1. ГРАФИКИ МЕТРИК ОБУЧЕНИЯ
print("\n📈 ГРАФИКИ МЕТРИК ОБУЧЕНИЯ (loss, mAP, precision, recall)")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'results.png')))

# 2. CONFUSION MATRICES
print("\n🎯 CONFUSION MATRIX")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'confusion_matrix.png')))

print("\n🎯 NORMALIZED CONFUSION MATRIX")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'confusion_matrix_normalized.png')))

# 3. F1, PRECISION, RECALL КРИВЫЕ
print("\n📊 F1-CONFIDENCE CURVE")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'BoxF1_curve.png')))

print("\n📊 PRECISION-CONFIDENCE CURVE")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'BoxP_curve.png')))

print("\n📊 RECALL-CONFIDENCE CURVE")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'BoxR_curve.png')))

print("\n📊 PRECISION-RECALL CURVE")
print("-"*70)
display(Image(filename=os.path.join(results_path, 'BoxPR_curve.png')))

# 4. ПРИМЕРЫ ВАЛИДАЦИИ (первые 3 батча)
for i in range(3):
    print(f"\n🖼️ ВАЛИДАЦИОННЫЙ БАТЧ {i} - ПРЕДСКАЗАНИЯ")
    print("-"*70)
    display(Image(filename=os.path.join(results_path, f'val_batch{i}_pred.jpg')))

    print(f"\n🏷️ ВАЛИДАЦИОННЫЙ БАТЧ {i} - GROUND TRUTH")
    print("-"*70)
    display(Image(filename=os.path.join(results_path, f'val_batch{i}_labels.jpg')))

# 5. CSV С ДЕТАЛЬНЫМИ МЕТРИКАМИ
print("\n" + "="*70)
print("📋 ДЕТАЛЬНЫЕ МЕТРИКИ ИЗ CSV")
print("="*70)

csv_path = os.path.join(results_path, 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    print("\n📊 Последние 10 эпох:")
    print(df.tail(10).to_string(index=False))

    print("\n\n📈 Лучшие результаты:")
    print(f"   🏆 Лучший mAP50: {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax()})")
    print(f"   🏆 Лучший mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax()})")
    print(f"   🏆 Минимальный box_loss: {df['train/box_loss'].min():.4f} (epoch {df['train/box_loss'].idxmin()})")
    print(f"   🏆 Минимальный cls_loss: {df['train/cls_loss'].min():.4f} (epoch {df['train/cls_loss'].idxmin()})")

    print("\n\n📉 Динамика основных метрик:")
    key_epochs = [0, 5, 10, 15, 20, 25, 29]  # ключевые эпохи
    cols_to_show = ['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
    print(df.iloc[key_epochs][cols_to_show].to_string(index=False))
else:
    print("❌ CSV файл не найден")

print("\n" + "="*70)
print("✅ ВИЗУАЛИЗАЦИЯ ЗАВЕРШЕНА")
print("="*70)